# 🐝 TTC: Insects 50:50 SupMinority (Ours) [Table 1]
Tự động chuẩn bị Dataset iNat21, Pretrain 350 Epochs, Linear Probe 50 Epochs, lưu Checkpoint vào `/kaggle/working/` và log WandB.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Clone mã nguồn TTC
import os
if not os.path.exists('TTC'):
    !git clone https://github.com/XiaoSha59/TTC.git
%cd TTC
!git pull

In [ ]:
# 3. Cài đặt các thư viện
!pip install -q 'lightning>=2.0.0' 'hydra-core>=1.3.2' omegaconf pyrootutils timm

In [ ]:
# 4. Cấu hình WandB (Tùy chọn: có thể thay bằng key riêng nếu muốn)
import os, wandb
os.environ['WANDB_API_KEY'] = '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'
!wandb login $WANDB_API_KEY

In [ ]:
# 5. Thiết lập Dataset iNat21 Natural (Tự động nhận diện hoặc tải nhanh từ AWS S3)
import os, glob, subprocess, shutil
os.makedirs('data/inat21', exist_ok=True)

# Kiểm tra train_mini trong input
train_cands = glob.glob('/kaggle/input/**/train_mini', recursive=True)
if train_cands:
    train_src = train_cands[0]
    print(f'>>> Tìm thấy train_mini có sẵn: {train_src}')
else:
    print('>>> Đang tải nhanh train_mini từ AWS S3...')
    !wget -q --show-progress -O /tmp/train_mini.tar.gz https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz
    !tar -xzf /tmp/train_mini.tar.gz -C /tmp/
    !rm -f /tmp/train_mini.tar.gz
    train_src = '/tmp/train_mini'

if not os.path.exists('data/inat21/train_mini'):
    os.symlink(train_src, 'data/inat21/train_mini')
if not os.path.exists('data/inat21/train'):
    os.symlink(train_src, 'data/inat21/train')

# Tải tập val chuẩn (8.3GB)
if not os.path.exists('/tmp/val'):
    print('>>> Đang tải tập val từ AWS S3...')
    !wget -q --show-progress -O /tmp/val.tar.gz https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz
    print('>>> Đang giải nén tập val vào /tmp...')
    !tar -xzf /tmp/val.tar.gz -C /tmp/
    !rm -f /tmp/val.tar.gz

if not os.path.exists('data/inat21/val'):
    os.symlink('/tmp/val', 'data/inat21/val')

# Kiểm tra số lượng ảnh
from data.iNatData import INaturalistNClasses
t_ds = INaturalistNClasses('data/inat21', split='train', classes=['Animalia_Arthropoda_Insecta_Hymenoptera_Apidae', 'Animalia_Arthropoda_Insecta_Hymenoptera_Vespidae'])
v_ds = INaturalistNClasses('data/inat21', split='val', classes=['Animalia_Arthropoda_Insecta_Hymenoptera_Apidae', 'Animalia_Arthropoda_Insecta_Hymenoptera_Vespidae'])
print(f'✅ Dataset iNat21 sẵn sàng! Train: {len(t_ds)} ảnh, Val: {len(v_ds)} ảnh')

In [ ]:
# 6. [GIAI ĐOẠN 1] Pre-training 350 Epochs SupMinority 50:50
print('🚀 [1/2] BẮT ĐẦU PRE-TRAIN 350 EPOCHS SUP-MINORITY 50:50...')
!python train.py \
    experiment=contrastive \
    experiment/specs=insects \
    class_ratios=[0.5,0.5] \
    module.ratio_supervised_majority=0.0 \
    batch_size=256 \
    trainer.max_epochs=350 \
    module.lr=0.0625 \
    trainer.precision=16-mixed \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    trainer.check_val_every_n_epoch=5 \
    name='insects-50_50-supmin-350ep-full'

In [ ]:
# 7. [GIAI ĐOẠN 2] Linear Probing 50 Epochs và Sao lưu Checkpoint
import glob, os, shutil
ckpts = sorted(glob.glob('logs/train/runs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError('Không tìm thấy last.ckpt!')
last_ckpt = ckpts[-1]
print(f'>>> Checkpoint backbone: {last_ckpt}')

# Copy checkpoint sang /kaggle/working/ để lưu vĩnh viễn trong Output
os.makedirs('/kaggle/working/saved_checkpoints', exist_ok=True)
saved_backbone = '/kaggle/working/saved_checkpoints/insects_50_50_supmin_350ep_backbone.ckpt'
shutil.copyfile(last_ckpt, saved_backbone)
print(f'✅ Đã sao lưu Checkpoint tại: {saved_backbone} ({os.path.getsize(saved_backbone)/1e6:.1f} MB)')

# Chạy Linear Probe 50 epochs
print('🚀 [2/2] BẮT ĐẦU LINEAR PROBING 50 EPOCHS...')
!python train.py \
    experiment=finetune \
    experiment/specs=insects \
    +base_model_path={saved_backbone} \
    trainer.max_epochs=50 \
    module.optimizer_name=adam \
    module.lr=0.001 \
    train_transform._target_=data.augmentation.SimCLRValTransform \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    name='insects-50_50-supmin-official-probe'

In [ ]:
# 8. Tổng kết
print('🎉🎉🎉 HOÀN THÀNH XUẤT SẮC 50:50 SUP-MINORITY!')
!ls -lh /kaggle/working/saved_checkpoints